# Notebook 4 — FiLM-Conditioned RoBERTa Baseline

This notebook tests a stronger subgroup-conditioning mechanism: **FiLM conditioning**.

Previous models provided subgroup identity either:

1. as a text prefix, or  
2. as an embedding concatenated to the final RoBERTa representation.

Both approaches allow the model to mostly ignore subgroup identity.

FiLM changes this by using subgroup identity to modulate the RoBERTa representation before prediction:

```text
h_film = gamma(subgroup) * h_text + beta(subgroup)
```

where:

- `h_text` is the RoBERTa CLS representation,
- `gamma(subgroup)` scales the text features,
- `beta(subgroup)` shifts the text features.

This notebook keeps the same data, splits, KL loss, and evaluation setup as the previous subgroup-aware baseline.


## 1. Imports and Configuration

In [3]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3
MAX_LENGTH = 192
BATCH_SIZE = 16
EPOCHS = 6
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
FILM_HIDDEN_DIM = 128
DROPOUT = 0.1
FILM_STRENGTH = 2.0
# Change this to "women" or "immigration"
EXPERIMENT = "immigration"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/data/processed")
OUTPUT_DIR = Path("boostedfilm_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Experiment:", EXPERIMENT)


Device: cuda
Experiment: immigration


In [4]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Processed Data

In [5]:
women_df = pd.read_parquet(DATA_DIR / "women_processed.parquet")
immigration_df = pd.read_parquet(DATA_DIR / "immigration_processed.parquet")

print("Women:", women_df.shape)
print("Immigration:", immigration_df.shape)

display(women_df.head(2))
display(immigration_df.head(2))


Women: (3869, 12)
Immigration: (3799, 12)


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,6,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),train,2,"[2.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0.0,0,0.0,"{'men': 1, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}","{'men': [1.0, 0.0, 0.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [1.0, 0.0, 0.0]}"
1,11,"eat my fuck, bitch",validation,2,"[0.0, 1.0, 1.0]","[0.0, 0.5, 0.5]",1.0,1,1.5,"{'men': 1, 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': 1}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}","{'men': [0.0, 0.0, 1.0], 'non_binary': None, 'prefer_not_to_say': None, 'self_describe': None, 'women': [0.0, 1.0, 0.0]}"


,comment_id,text,split,num_annotations,overall_counts,overall_distribution,entropy,majority_label,expected_label,subgroup_counts,subgroup_label_counts,subgroup_distributions
0,7,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,test,2,"[1.0, 0.0, 1.0]","[0.5, 0.0, 0.5]",1.000000,0,1.000000,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': 1.0, 'liberal': 1.0, 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': [1.0, 0.0, 0.0], 'liberal': [0.0, 0.0, 1.0], 'neutral': None, 'no_opinion': None, 'slightly_conservative': None, 'slightly_liberal': None, '..."
1,10,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...",train,3,"[1.0, 0.0, 2.0]","[0.3333333432674408, 0.0, 0.6666666865348816]",0.918296,2,1.333333,"{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': 1.0, 'neutral': 1.0, 'no_opinion': None, 'slightly_conservative': 1.0, 'slightly_liberal': None, 'unknown': None}","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera...","{'conservative': None, 'extremely_conservative': None, 'extremely_liberal': None, 'liberal': [0.0, 0.0, 1.0], 'neutral': [0.0, 0.0, 1.0], 'no_opinion': None, 'slightly_conservative': [1.0, 0.0, 0.0], 'slightly_libera..."


## 3. Expand Comment Rows into Comment–Subgroup Examples

In [6]:
def ensure_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        return ast.literal_eval(value)
    raise TypeError(f"Expected dict or stringified dict, got {type(value)}")


def is_valid_distribution(dist, num_labels: int = NUM_LABELS, tolerance: float = 1e-5) -> bool:
    if dist is None:
        return False
    try:
        dist = [float(x) for x in dist]
    except Exception:
        return False
    if len(dist) != num_labels:
        return False
    if any(p < -tolerance for p in dist):
        return False
    return abs(sum(dist) - 1.0) < tolerance


def expand_subgroup_examples(
    comment_df: pd.DataFrame,
    experiment_name: str,
    allowed_subgroups: list[str] | None = None,
) -> pd.DataFrame:
    rows = []

    for _, row in comment_df.iterrows():
        subgroup_distributions = ensure_dict(row["subgroup_distributions"])
        subgroup_counts = ensure_dict(row["subgroup_counts"])

        for subgroup, target_distribution in subgroup_distributions.items():
            if allowed_subgroups is not None and subgroup not in allowed_subgroups:
                continue

            if not is_valid_distribution(target_distribution):
                continue

            target_distribution = [float(x) for x in target_distribution]

            rows.append({
                "experiment": experiment_name,
                "comment_id": row["comment_id"],
                "split": row["split"],
                "subgroup": subgroup,
                "subgroup_count": int(subgroup_counts.get(subgroup, 0)),
                "text": row["text"],
                "target_distribution": target_distribution,
                "target_majority_label": int(np.argmax(target_distribution)),
                "target_expected_label": float(np.dot(np.arange(NUM_LABELS), target_distribution)),
                "target_entropy": float(entropy(target_distribution, base=2)),
            })

    return pd.DataFrame(rows)


In [7]:
WOMEN_ALLOWED_SUBGROUPS = ["women", "men"]
Immigration_ALLOWED_SUBGROUPS = ["neutral","no_opinion","liberal","conservative","slightly_liberal","slightly_conservative","extremely_liberal","extremely_conservative"]
women_examples = expand_subgroup_examples(
    women_df,
    experiment_name="women",
    allowed_subgroups=WOMEN_ALLOWED_SUBGROUPS,
)

immigration_examples = expand_subgroup_examples(
    immigration_df,
    experiment_name="immigration",
    allowed_subgroups=Immigration_ALLOWED_SUBGROUPS,
)

print("Women examples:", women_examples.shape)
print("Immigration examples:", immigration_examples.shape)

display(women_examples.head())
display(immigration_examples.head())


Women examples: (7738, 10)
Immigration examples: (9090, 10)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,women,6,train,men,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
1,women,6,train,women,1,First off you look cool as fuck! Anyway if we were in the bedroom I think I'd cream your ass then make you suck my cock clean like a whore ;),"[1.0, 0.0, 0.0]",0,0.0,0.0
2,women,11,validation,men,1,"eat my fuck, bitch","[0.0, 0.0, 1.0]",2,2.0,0.0
3,women,11,validation,women,1,"eat my fuck, bitch","[0.0, 1.0, 0.0]",1,1.0,0.0
4,women,26,train,men,2,I'd love to rip those panties off and shove my fat cock in her ass.,"[0.5, 0.0, 0.5]",0,1.0,1.0


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,0.0
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,0.0
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[1.0, 0.0, 0.0]",0,0.0,0.0


## 4. Select Experiment and Create Subgroup IDs

In [8]:
if EXPERIMENT == "women":
    model_df = women_examples.copy()
elif EXPERIMENT == "immigration":
    model_df = immigration_examples.copy()
else:
    raise ValueError("EXPERIMENT must be 'women' or 'immigration'.")

subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(model_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

model_df["subgroup_id"] = model_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

display(pd.crosstab(model_df["split"], model_df["subgroup"]))
display(model_df.head())


Subgroup mapping:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,0.0,2
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,0.0,3
2,immigration,10,train,liberal,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0,3
3,immigration,10,train,neutral,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[0.0, 0.0, 1.0]",2,2.0,0.0,4
4,immigration,10,train,slightly_conservative,1,"They'll come back in your plan, also. Plus we pay to jail them for 2 years. With my plan, we jail them after they come back. And deport them again. It really doesn't matter either way until we're build the wall, and ...","[1.0, 0.0, 0.0]",0,0.0,0.0,6


In [9]:
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
val_df = model_df[model_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = model_df[model_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

print("Training majority-label distribution:")
display(train_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("Test majority-label distribution:")
display(test_df["target_majority_label"].value_counts(normalize=True).sort_index())


Train: (6360, 11)
Val: (1346, 11)
Test: (1384, 11)
Training majority-label distribution:


target_majority_label
0    0.731604
1    0.070912
2    0.197484
Name: proportion, dtype: float64

Test majority-label distribution:


target_majority_label
0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

## 5. Dataset and Dataloader

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [11]:
class SubgroupDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(row["target_distribution"], dtype=torch.float),
        }


In [12]:
train_dataset = SubgroupDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = SubgroupDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = SubgroupDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([16, 192]),
 'attention_mask': torch.Size([16, 192]),
 'subgroup_id': torch.Size([16]),
 'target_distribution': torch.Size([16, 3])}

## 6. FiLM-Conditioned RoBERTa Model

In [ ]:
class FiLMConditionedRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        film_hidden_dim: int = 128,
        num_labels: int = 3,
        dropout: float = 0.1,
        film_strength: float = 2.0
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.film_strength = 2.0
        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.film_generator = nn.Sequential(
            nn.Linear(subgroup_dim, film_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(film_hidden_dim, hidden_size * 2),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        film_params = self.film_generator(subgroup_embedding)
        gamma, beta = film_params.chunk(2, dim=-1)

        
        gamma = 1.0 + self.film_strength * torch.tanh(gamma)
        beta = self.film_strength * torch.tanh(beta)

        film_embedding = gamma * cls_embedding + beta

        logits = self.classifier(film_embedding)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
            "gamma": gamma,
            "beta": beta,
        }


In [14]:
model = FiLMConditionedRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    film_hidden_dim=FILM_HIDDEN_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_training_steps = len(train_loader) * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/shayan/Distributional-Hate-Speech-Prediction/.venv/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training steps: 2388
Warmup steps: 238


## 7. Metrics

In [15]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 8. Training Helpers

In [16]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            loss = criterion(outputs["log_probs"], targets)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        total_loss += loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 9. Train Model

In [17]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / f"{EXPERIMENT}_film_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / f"{EXPERIMENT}_film_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/6
Train KL: 0.7115 | Val KL: 0.6211
Train JS: 0.2805 | Val JS: 0.2216
Val Acc: 0.7682 | Val Macro F1: 0.4040
Saved new best model to boostedfilm_outputs/immigration_film_best_model.pt
Epoch 2/6
Train KL: 0.5857 | Val KL: 0.6088
Train JS: 0.2209 | Val JS: 0.2131
Val Acc: 0.7764 | Val Macro F1: 0.4574
Saved new best model to boostedfilm_outputs/immigration_film_best_model.pt
Epoch 3/6
Train KL: 0.5371 | Val KL: 0.6141
Train JS: 0.1999 | Val JS: 0.2124
Val Acc: 0.7779 | Val Macro F1: 0.4703
Epoch 4/6
Train KL: 0.4815 | Val KL: 0.6432
Train JS: 0.1810 | Val JS: 0.2118
Val Acc: 0.7719 | Val Macro F1: 0.4672


KeyboardInterrupt: 

## 10. Test Evaluation

In [18]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_film_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_41678/2391530061.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6101340055465698,
 'js_mean': 0.21948093922700418,
 'cross_entropy_mean': 0.6345587968826294,
 'accuracy': 0.7774566473988439,
 'macro_f1': 0.4617669310920845,
 'expected_label_mae': 0.472862783666684,
 'entropy_pearson': 0.033174569507014535,
 'entropy_spearman': 0.033593383307644255,
 'loss': 0.61013400399616}

Saved: boostedfilm_outputs/immigration_film_test_metrics.json


## 11. Save Predictions

In [19]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / f"{EXPERIMENT}_film_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

print("Saved:", predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,target_entropy,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,0.0,2,"[1.0, 0.0, 0.0]","[0.3518709, 0.13339679, 0.51473236]",2,1.162862,1.411074,1.044491,0.440883,1.044491
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,0.0,3,"[0.0, 0.0, 1.0]","[0.3588195, 0.110478975, 0.5307015]",2,1.171882,1.366771,0.633556,0.287389,0.633556
2,immigration,66,test,extremely_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[0.0, 0.0, 1.0]",2,2.0,0.0,2,"[0.0, 0.0, 1.0]","[0.13399403, 0.075266525, 0.7907394]",2,1.656745,0.937277,0.234787,0.113470,0.234787
3,immigration,66,test,slightly_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[1.0, 0.0, 0.0]",0,0.0,0.0,7,"[1.0, 0.0, 0.0]","[0.14117818, 0.05938065, 0.79944116]",2,1.658263,0.898822,1.957733,0.691916,1.957733
4,immigration,108,test,conservative,1,"I am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportations but look what happened when he threatened to do it the first time, we got flooded with those lazy fu...","[0.0, 0.0, 1.0]",2,2.0,0.0,0,"[0.0, 0.0, 1.0]","[0.26275134, 0.06410698, 0.6731417]",2,1.410390,1.145101,0.395799,0.186608,0.395799


Saved: boostedfilm_outputs/immigration_film_test_predictions.parquet


## 12. Standard Diagnostics

In [20]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[950   0  79]
 [ 71   0  24]
 [134   0 126]]

Classification report:
              precision    recall  f1-score   support

           0       0.82      0.92      0.87      1029
           1       0.00      0.00      0.00        95
           2       0.55      0.48      0.52       260

    accuracy                           0.78      1384
   macro avg       0.46      0.47      0.46      1384
weighted avg       0.71      0.78      0.74      1384


Predicted label distribution:


0    0.834538
2    0.165462
Name: proportion, dtype: float64


True label distribution:


0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

In [21]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_entropy", "mean"),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,1029,0.261329,0.112235,0.035754,0.673598
1,95,2.603847,0.791253,0.052632,0.880638
2,260,1.262125,0.435010,0.026836,1.070538


Average predicted distribution by true majority label:
0 [0.81736106 0.06212457 0.12051498]
1 [0.6588862  0.07737466 0.26373917]
2 [0.49735343 0.08837496 0.4142717 ]


In [23]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / f"{EXPERIMENT}_film_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.402275,0.164670,0.412820,0.869565,0.555473,0.385791,0.036116,0.028072
0,conservative,163,0.483321,0.170871,0.495252,0.828221,0.511685,0.365159,-0.067703,-0.051285
5,no_opinion,46,0.499141,0.181370,0.518061,0.826087,0.540932,0.382993,-0.132682,-0.109944
6,slightly_conservative,154,0.569174,0.206335,0.580733,0.785714,0.456205,0.454974,-0.066717,-0.062195
3,liberal,308,0.582607,0.216347,0.622869,0.795455,0.477044,0.470300,0.025002,0.035259
7,slightly_liberal,213,0.665251,0.236316,0.682397,0.732394,0.422475,0.507564,0.042617,0.019140
4,neutral,257,0.673818,0.236236,0.707241,0.770428,0.453082,0.507779,0.120867,0.109224
2,extremely_liberal,197,0.721895,0.256516,0.742613,0.725888,0.407131,0.538215,-0.010654,-0.002215


Saved: boostedfilm_outputs/immigration_film_subgroup_metrics.csv


## 13. Counterfactual Subgroup Sensitivity

In [24]:
@torch.no_grad()
def predict_distribution(text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long,
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id,
    )

    return outputs["probs"].detach().cpu().numpy()[0]


In [25]:
subgroups = list(subgroup_to_id.keys())

counterfactual_rows = []

for _, row in test_df.drop_duplicates("comment_id").iterrows():
    text = row["text"]

    preds_by_group = {
        subgroup: predict_distribution(text, subgroup)
        for subgroup in subgroups
    }

    pairwise_js = []

    for group_a, group_b in itertools.combinations(subgroups, 2):
        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2,
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": row["comment_id"],
        "text": text,
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(
    cf_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

cf_path = OUTPUT_DIR / f"{EXPERIMENT}_film_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Mean counterfactual JS: 0.001971986951734781
Median counterfactual JS: 0.0016899239526931814
Mean max-pairwise JS: 0.006478903929617757
Median max-pairwise JS: 0.005278321888955764


,comment_id,text,mean_pairwise_js,max_pairwise_js
49,6953,"Fuck Russia, fucking pieces of shit.",0.004427,0.016232
68,8789,And 10 Pakistani street shit eaters,0.004460,0.016032
2,108,"I am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportations but look what happened when he threatened to do it the first time, we got flooded with those lazy fu...",0.004504,0.015935
452,43980,Trump is right - these maggots of color should gtfo,0.004578,0.015832
150,18971,Lmao trash ass commies. aMeRiCaNS fAT AmURIcAn sTuPId.,0.004402,0.015565
483,45185,Shoot a few of those border jumpers and the word will get out. Fuck them. Remember the Alamo!!!,0.004007,0.015326
555,48620,Go Islam destroy Whole Amirca,0.004078,0.015174
544,48149,"there is so much anti American scum running around. And you can see the white race destroying itself,",0.004144,0.015113
530,47662,more attacks on Americans by the traitor in chief all to the glory of stupid white trash,0.004418,0.015048
474,44946,Those four depraved fools are the faces of evil ! I agree with President Trump since they hate America so much go ahead and leave ! Omar Somalia is such a lovely peaceful Islamic place what are you doing in America ?...,0.003735,0.015031


Saved: boostedfilm_outputs/immigration_film_counterfactual_js.csv


## 14. Pairwise Counterfactual Analysis

In [26]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for _, row in test_df.drop_duplicates("comment_id").iterrows():
        text = row["text"]

        pred_a = predict_distribution(text, group_a)
        pred_b = predict_distribution(text, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2,
        ) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": text,
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
        })

    return pd.DataFrame(rows)


print("Available subgroups:")
print(subgroup_to_id)


Available subgroups:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}


In [27]:
if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    extreme_df = pairwise_counterfactual_js(
        "extremely_liberal",
        "extremely_conservative",
    )

    print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
    print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

    display(
        extreme_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    extreme_path = OUTPUT_DIR / f"{EXPERIMENT}_film_EL_vs_EC_counterfactual.csv"
    extreme_df.to_csv(extreme_path, index=False)
    print("Saved:", extreme_path)

if "men" in subgroup_to_id and "women" in subgroup_to_id:
    gender_df = pairwise_counterfactual_js("men", "women")

    print("Men vs women mean JS:", gender_df["js"].mean())
    print("Men vs women median JS:", gender_df["js"].median())

    display(
        gender_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    gender_path = OUTPUT_DIR / f"{EXPERIMENT}_film_men_vs_women_counterfactual.csv"
    gender_df.to_csv(gender_path, index=False)
    print("Saved:", gender_path)


Extreme liberal vs extreme conservative mean JS: 0.003488701541989004
Extreme liberal vs extreme conservative median JS: 0.0016537880258774948


,comment_id,text,group_a,group_b,js,pred_extremely_liberal,pred_extremely_conservative
44,6099,russian + racist + absolutely no life whatsoever + angry virgin take your pick,extremely_liberal,extremely_conservative,0.013030,"[0.57122785, 0.14313197, 0.2856402]","[0.4915058, 0.09918263, 0.40931162]"
234,29680,This guy is supposed to be the liberal face of Islamic State of Pakistan...this idiot barrister won't say how 210 million Paki Mujahids got their butts kicked by Windies and Australia who are kafirs & a fraction of p...,extremely_liberal,extremely_conservative,0.011666,"[0.5984134, 0.14365578, 0.25793087]","[0.5358916, 0.09617594, 0.3679325]"
197,24462,Better for Mexico keep this migrant out of our borders. USA right now is lauded of migrant so we know DS NDP Radical Communist Snakes don't give a sheet about that. But the hard working people of this country care a ...,extremely_liberal,extremely_conservative,0.011622,"[0.5581568, 0.1532922, 0.28855094]","[0.49797523, 0.101612754, 0.40041205]"
406,42209,Maybe he should send his sleazy wife and her chain migration communist parents back home. Just because she gives a brilliant blow job doesn't qualify her for that Einstein VISA.,extremely_liberal,extremely_conservative,0.011544,"[0.5804459, 0.14527343, 0.27428067]","[0.5066613, 0.103881635, 0.3894571]"
203,25122,"what kinda dumb fucking hole did u crawl out from japan was literally asian nazi germany they did genocide, mass rape, torture, human experimentation, all the works r u honestly saying you'd be happy if a world-famou...",extremely_liberal,extremely_conservative,0.011430,"[0.5477279, 0.14436533, 0.3079068]","[0.48058882, 0.09710817, 0.42230308]"
286,33753,"Like, she's FROM Somalia. The shear arrogance to whitemansplain to her, as if he knows her country of origin – THAT SHE FLED FROM – better than her. Lord, that man's a twat.",extremely_liberal,extremely_conservative,0.011387,"[0.5321316, 0.15123114, 0.31663728]","[0.4705306, 0.09981832, 0.4296511]"
238,29892,You just as they say send her back how bout we send you're slut wife back,extremely_liberal,extremely_conservative,0.011312,"[0.40413874, 0.16083163, 0.4350296]","[0.34767547, 0.101568304, 0.5507562]"
63,8333,"America would invade Sweden if they could cause ""muh socialism"". But they usually just content themselves with picking on vulnerable nonwhite countries like the cowardly inept bullies they are",extremely_liberal,extremely_conservative,0.011251,"[0.6296638, 0.1440663, 0.22626992]","[0.5696052, 0.09920422, 0.33119062]"
384,41270,I like Japan you twit. Let me help you that sentence . WHITE American citizens don't give a crap about Japanese citizens abducted by North Korea. The Same White Americans who voted this Orange Shit Stain as our presi...,extremely_liberal,extremely_conservative,0.011165,"[0.5021318, 0.14955923, 0.34830895]","[0.44087255, 0.097478315, 0.46164915]"
409,42283,We are laughing at the inbred Brits and the USA will not do anything because fighting for that shit inbred island is not worth it,extremely_liberal,extremely_conservative,0.011031,"[0.47196704, 0.14906625, 0.3789667]","[0.41189834, 0.095752075, 0.49234954]"


Saved: boostedfilm_outputs/immigration_film_EL_vs_EC_counterfactual.csv


## 15. True Subgroup Disagreement vs Model Counterfactual Disagreement

In [28]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(x, dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement(raw_df: pd.DataFrame, group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    for _, row in raw_df.iterrows():
        subgroup_dists = row["subgroup_distributions"]

        if isinstance(subgroup_dists, str):
            subgroup_dists = ast.literal_eval(subgroup_dists)

        dist_a = subgroup_dists.get(group_a)
        dist_b = subgroup_dists.get(group_b)

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": row["comment_id"],
            "text": row["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


In [29]:
if EXPERIMENT == "immigration" and "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    true_df = true_pairwise_disagreement(
        immigration_df,
        "extremely_liberal",
        "extremely_conservative",
    )

    comparison_df = true_df.merge(
        extreme_df[[
            "comment_id",
            "js",
            "pred_extremely_liberal",
            "pred_extremely_conservative",
        ]],
        on="comment_id",
        how="inner",
    )

    comparison_df = comparison_df.rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true EL/EC overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())
    print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
    print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["extremely_liberal_true_label"].astype(str)
        + "-"
        + comparison_df["extremely_conservative_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(
        comparison_df
        .sort_values("true_js", ascending=False)
        .head(30)
    )

    comparison_path = OUTPUT_DIR / f"{EXPERIMENT}_film_true_vs_model_EL_EC.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


N true EL/EC overlap: 101
N comparison: 8
Mean true JS: 0.16448066401642414
Mean model JS: 0.005363110322547998
Mean capture ratio, true_js > 0: 0.3947584016513304
Median capture ratio, true_js > 0: 0.013722958269014839


,n,mean_true_js,mean_model_js
label_pair,,,
0-1,1,1.000000,0.009841
0-0,4,0.077820,0.004090
2-2,3,0.001522,0.005567


,comment_id,text,true_js,extremely_liberal_true_dist,extremely_conservative_true_dist,extremely_liberal_true_label,extremely_conservative_true_label,model_js,pred_extremely_liberal,pred_extremely_conservative,label_pair
7,49332,Send all these asses back !,1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.009841,"[0.5140247, 0.1380334, 0.34794188]","[0.4511715, 0.093249336, 0.45557916]",0-1
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,0.311278,"[0.5, 0.0, 0.5]","[1.0, 0.0, 0.0]",0,0,0.004272,"[0.7946909, 0.10500579, 0.100303285]","[0.77177244, 0.08189106, 0.14633642]",0-0
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites ou...,0.004567,"[0.08421052992343903, 0.10526315867900848, 0.8105263113975525]","[0.06666667014360428, 0.06666667014360428, 0.8666666746139526]",2,2,0.005301,"[0.17445001, 0.095983215, 0.72956675]","[0.14956559, 0.056907766, 0.79352665]",2-2
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.003867,"[0.14844789, 0.092573985, 0.7589782]","[0.13092758, 0.058197565, 0.8108748]",2-2
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.000790,"[0.93147045, 0.04766308, 0.020866355]","[0.9388903, 0.035718642, 0.025391055]",0-0
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.006056,"[0.65755916, 0.16220461, 0.18023625]","[0.63183504, 0.12143489, 0.24673004]",0-0
5,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupation forces roaming t...,0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.007534,"[0.28422558, 0.1086776, 0.60709685]","[0.24216364, 0.0639412, 0.6938952]",2-2
6,43958,"Donald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don't matter who you are fuc...",0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.005244,"[0.69510657, 0.13893808, 0.16595532]","[0.6742129, 0.10212718, 0.22365998]",0-0


Saved: boostedfilm_outputs/immigration_film_true_vs_model_EL_EC.csv


## 16. Interpretation Guide

Compare this FiLM model against:

1. the subgroup-token baseline,
2. the subgroup-embedding baseline,
3. the ordinal-loss run.

The most important questions are:

```text
1. Does overall KL/JS improve?
2. Does class 1 appear as a predicted class?
3. Does class 2 become less suppressed?
4. Does counterfactual subgroup sensitivity increase?
5. Does model EL↔EC JS move closer to true EL↔EC JS?
```

If FiLM increases counterfactual subgroup sensitivity even without improving overall KL, it still supports the claim that stronger identity conditioning changes model behaviour.
If FiLM also fails, that is strong evidence that the remaining disagreement requires richer contextual information rather than simple subgroup identifiers.
